# Bike Sharing in the Bay Area

We end this chapter by using all the methods we have learned to examine a new and large dataset. We will also draw maps of the bike stations.

The [Bay Area Bike Share](http://www.bayareabikeshare.com/) service published a [dataset](http://www.bayareabikeshare.com/open-data) describing every bicycle rental from September 2014 to August 2015 in their system. There were 354,152 rentals in all. The columns are:

- An ID for the rental
- Duration of the rental, in seconds
- Start date
- Name of the Start Station and code for Start Terminal
- Name of the End Station and code for End Terminal
- A serial number for the bike
- Subscriber type and zip code

In [1]:
import polars as pl
import hvplot.polars
from IPython.display import display
path_data = '../../../assets/data/'
import math
from scipy import stats
import folium


In [2]:
# POLARS_CHANGE: some zip codes are the text "nil", so that column is read as strings
trips = pl.read_csv(path_data + "trip.csv", schema_overrides={"Zip Code": pl.String})
trips


Trip ID,Duration,Start Date,Start Station,Start Terminal,End Date,End Station,End Terminal,Bike #,Subscriber Type,Zip Code
i64,i64,str,str,i64,str,str,i64,i64,str,str
913460,765,"""8/31/2015 23:26""","""Harry Bridges Plaza (Ferry Bui…",50,"""8/31/2015 23:39""","""San Francisco Caltrain (Townse…",70,288,"""Subscriber""","""2139"""
913459,1036,"""8/31/2015 23:11""","""San Antonio Shopping Center""",31,"""8/31/2015 23:28""","""Mountain View City Hall""",27,35,"""Subscriber""","""95032"""
913455,307,"""8/31/2015 23:13""","""Post at Kearny""",47,"""8/31/2015 23:18""","""2nd at South Park""",64,468,"""Subscriber""","""94107"""
913454,409,"""8/31/2015 23:10""","""San Jose City Hall""",10,"""8/31/2015 23:17""","""San Salvador at 1st""",8,68,"""Subscriber""","""95113"""
913453,789,"""8/31/2015 23:09""","""Embarcadero at Folsom""",51,"""8/31/2015 23:22""","""Embarcadero at Sansome""",60,487,"""Customer""","""9069"""
…,…,…,…,…,…,…,…,…,…,…
432951,619,"""9/1/2014 4:21""","""Powell Street BART""",39,"""9/1/2014 4:32""","""Townsend at 7th""",65,335,"""Subscriber""","""94118"""
432950,6712,"""9/1/2014 3:16""","""Harry Bridges Plaza (Ferry Bui…",50,"""9/1/2014 5:08""","""San Francisco Caltrain (Townse…",70,259,"""Customer""","""44100"""
432949,538,"""9/1/2014 0:05""","""South Van Ness at Market""",66,"""9/1/2014 0:14""","""5th at Howard""",57,466,"""Customer""","""32"""


We'll focus only on the *free trips*, which are trips that last less than 1800 seconds (half an hour). There is a charge for longer trips.

The histogram below shows that most of the trips took around 10 minutes (600 seconds) or so. Very few took near 30 minutes (1800 seconds), possibly because people try to return the bikes before the cutoff time so as not to have to pay.

In [3]:
commute = trips.filter(pl.col("Duration") < 1800)
commute.hvplot.hist(
    "Duration",
    bins=10,
    normed=True,
    xlabel="Duration (Second)",
)

:Histogram   [Duration]   (Frequency)

We can get more detail by specifying a larger number of bins. But the overall shape doesn't change much.

In [4]:
commute.hvplot.hist(
    "Duration",
    bins=60,
    normed=True,
    xlabel="Duration (Second)",
)


:Histogram   [Duration]   (Frequency)

## Exploring the Data with `group` and `pivot`

We can use `group` to identify the most highly used Start Station:

In [5]:
starts = commute.group_by("Start Station").len(name="count").sort("count", descending=True)
starts


Start Station,count
str,u32
"""San Francisco Caltrain (Townse…",25858
"""San Francisco Caltrain 2 (330 …",21523
"""Harry Bridges Plaza (Ferry Bui…",15543
"""Temporary Transbay Terminal (H…",14298
"""2nd at Townsend""",13674
…,…
"""Mezes Park""",189
"""Redwood City Medical Center""",139
"""San Mateo County Center""",108


The largest number of trips started at the Caltrain Station on Townsend and 4th in San Francisco. People take the train into the city, and then use a shared bike to get to their next destination.

The `group` method can also be used to classify the rentals by both Start Station and End Station.

In [6]:
commute.group_by(["Start Station", "End Station"]).len(name="count").sort("count", descending=True)


Start Station,End Station,count
str,str,u32
"""San Francisco Caltrain 2 (330 …","""Townsend at 7th""",3737
"""2nd at Townsend""","""Harry Bridges Plaza (Ferry Bui…",2947
"""Harry Bridges Plaza (Ferry Bui…","""Embarcadero at Sansome""",2729
"""Townsend at 7th""","""San Francisco Caltrain 2 (330 …",2723
"""Harry Bridges Plaza (Ferry Bui…","""2nd at Townsend""",2607
…,…,…
"""California Ave Caltrain Statio…","""Rengstorff Avenue / California…",1
"""San Antonio Shopping Center""","""Evelyn Park and Ride""",1
"""San Antonio Shopping Center""","""Cowper at University""",1


Fifty-four trips both started and ended at the station on 2nd at Folsom. A much large number (437) were between 2nd at Folsom and 2nd at Townsend. 

The `pivot` method does the same classification but displays its results in a contingency table that shows all possible combinations of Start and End Stations, even though some of them didn't correspond to any trips. Remember that the first argument of `pivot` specifies the column labels of the pivot table; `index` labels the rows.

There is a train station as well as a Bay Area Rapid Transit (BART) station near Beale at Market, explaining the high number of trips that start and end there.

#### Drawing Maps
The DataFrame `stations` contains geographical information about each bike station, including latitude, longitude, and a "landmark" which is the name of the city where the station is located.

In [7]:
stations = pl.read_csv(path_data + "station.csv")
stations


station_id,name,lat,long,dockcount,landmark,installation
i64,str,f64,f64,i64,str,str
2,"""San Jose Diridon Caltrain Stat…",37.329732,-121.901782,27,"""San Jose""","""8/6/2013"""
3,"""San Jose Civic Center""",37.330698,-121.888979,15,"""San Jose""","""8/5/2013"""
4,"""Santa Clara at Almaden""",37.333988,-121.894902,11,"""San Jose""","""8/6/2013"""
5,"""Adobe on Almaden""",37.331415,-121.8932,19,"""San Jose""","""8/5/2013"""
6,"""San Pedro Square""",37.336721,-121.894074,15,"""San Jose""","""8/7/2013"""
…,…,…,…,…,…,…
77,"""Market at Sansome""",37.789625,-122.400811,27,"""San Francisco""","""8/25/2013"""
80,"""Santa Clara County Civic Cente…",37.352601,-121.905733,15,"""San Jose""","""12/31/2013"""
82,"""Broadway St at Battery St""",37.798541,-122.400862,15,"""San Francisco""","""1/22/2014"""


We can draw a map of where the stations are located. The DataFrame's columns are, in order, latitude, longitude, and an optional identifier for each point.

In [8]:
# POLARS_CHANGE: Polars does not draw maps, so folium places one marker per station
points = stations.select("lat", "long", "name").rename({"name": "labels"})
station_map = folium.Map(
    location=[points.get_column("lat").mean(), points.get_column("long").mean()],
    zoom_start=10,
)
for lat, lon, name in points.iter_rows():
    folium.Marker([lat, lon], popup=name).add_to(station_map)
station_map


The map is created using [OpenStreetMap](http://www.openstreetmap.org/#map=5/51.500/-0.100), which is an open online mapping system that you can use just as you would use Google Maps or any other online map. Zoom in to San Francisco to see how the stations are distributed. Click on a marker to see which station it is.

You can also represent points on a map by colored circles. Here is such a map of the San Francisco bike stations.

In [9]:
sf = stations.filter(pl.col("landmark") == "San Francisco")
sf_map_data = sf.select("lat", "long", "name").rename({"name": "labels"})
sf_map = folium.Map(
    location=[sf_map_data.get_column("lat").mean(), sf_map_data.get_column("long").mean()],
    zoom_start=13,
)
for lat, lon, name in sf_map_data.iter_rows():
    folium.CircleMarker([lat, lon], popup=name, color="green", fill=True, radius=10).add_to(sf_map)
sf_map


## More Informative Maps: An Application of `join`
The bike stations are located in five different cities in the Bay Area. To distinguish the points by using a different color for each city, let's start by using group to identify all the cities and assign each one a color.

In [10]:
# POLARS_CHANGE: group_by does not sort, so cities are sorted before the colors are assigned
cities = stations.group_by("landmark").len(name="count").sort("landmark").rename({"landmark": "city"})
cities


city,count
str,u32
"""Mountain View""",7
"""Palo Alto""",5
"""Redwood City""",7
"""San Francisco""",35
"""San Jose""",16


In [11]:
colors = cities.with_columns(
    pl.Series("color", ["blue", "red", "green", "orange", "purple"])
)
colors


city,count,color
str,u32,str
"""Mountain View""",7,"""blue"""
"""Palo Alto""",5,"""red"""
"""Redwood City""",7,"""green"""
"""San Francisco""",35,"""orange"""
"""San Jose""",16,"""purple"""


Now we can join `stations` and `colors` by `landmark`, and then select the columns we need to draw a map.

In [12]:
joined = stations.join(colors, left_on="landmark", right_on="city")
colored = joined.select("lat", "long", "name", "color").rename({"name": "labels"})
colored_map = folium.Map(
    location=[colored.get_column("lat").mean(), colored.get_column("long").mean()],
    zoom_start=10,
)
for lat, lon, name, color in colored.iter_rows():
    folium.Marker([lat, lon], popup=name, icon=folium.Icon(color=color)).add_to(colored_map)
colored_map


Now the markers have five different colors for the five different cities.

To see where most of the bike rentals originate, let's identify the start stations:

In [13]:
starts = commute.group_by("Start Station").len(name="count").sort("count", descending=True)
starts


Start Station,count
str,u32
"""San Francisco Caltrain (Townse…",25858
"""San Francisco Caltrain 2 (330 …",21523
"""Harry Bridges Plaza (Ferry Bui…",15543
"""Temporary Transbay Terminal (H…",14298
"""2nd at Townsend""",13674
…,…
"""Mezes Park""",189
"""Redwood City Medical Center""",139
"""San Mateo County Center""",108


We can include the geographical data needed to map these stations, by first joining `starts` with `stations`:

In [14]:
station_starts = stations.join(starts, left_on="name", right_on="Start Station").sort("name")
station_starts


station_id,name,lat,long,dockcount,landmark,installation,count
i64,str,f64,f64,i64,str,str,u32
62,"""2nd at Folsom""",37.785299,-122.396236,19,"""San Francisco""","""8/22/2013""",7841
64,"""2nd at South Park""",37.782259,-122.392738,15,"""San Francisco""","""8/22/2013""",9274
61,"""2nd at Townsend""",37.780526,-122.390288,27,"""San Francisco""","""8/22/2013""",13674
57,"""5th at Howard""",37.781752,-122.405127,15,"""San Francisco""","""8/21/2013""",7394
5,"""Adobe on Almaden""",37.331415,-121.8932,19,"""San Jose""","""8/5/2013""",522
…,…,…,…,…,…,…,…
74,"""Steuart at Market""",37.794139,-122.394434,23,"""San Francisco""","""8/25/2013""",13215
55,"""Temporary Transbay Terminal (H…",37.789756,-122.394643,23,"""San Francisco""","""8/20/2013""",14298
65,"""Townsend at 7th""",37.771058,-122.402717,15,"""San Francisco""","""8/22/2013""",13579


Now we extract just the data needed for drawing our map, adding a color and an area to each station. The area is 0.3 times the count of the number of rentals starting at each station, where the constant 0.3 was chosen so that the circles would appear at an appropriate scale on the map.

In [15]:
starts_map_data = station_starts.select("lat", "long", "name").with_columns(
    pl.lit("blue").alias("colors"),
    (station_starts.get_column("count") * 0.3).alias("areas"),
).rename({"name": "labels"})
starts_map_data.show(3)
starts_map = folium.Map(
    location=[starts_map_data.get_column("lat").mean(), starts_map_data.get_column("long").mean()],
    zoom_start=10,
)
for lat, lon, name, color, area in starts_map_data.iter_rows():
    folium.CircleMarker(
        [lat, lon],
        popup=name,
        color=color,
        fill=True,
        radius=(area / math.pi) ** 0.5,
    ).add_to(starts_map)
starts_map


lat,long,labels,colors,areas
f64,f64,str,str,f64
37.785299,-122.396236,"""2nd at Folsom""","""blue""",2352.3
37.782259,-122.392738,"""2nd at South Park""","""blue""",2782.2
37.780526,-122.390288,"""2nd at Townsend""","""blue""",4102.2


That huge blob in San Francisco shows that the eastern section of the city is the unrivaled capital of bike rentals in the Bay Area.